In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
X_train = joblib.load("../data/X_train_raw.pkl")
X_val = joblib.load("../data/X_val_raw.pkl")
X_test = joblib.load("../data/X_test_raw.pkl")

y_train = joblib.load("../data/y_train.pkl")
y_val = joblib.load("../data/y_val.pkl")
y_test = joblib.load("../data/y_test.pkl")

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (184506, 121)
Validation: (61502, 121)
Test: (61503, 121)


In [3]:
def create_application_features(df):
    df = df.copy()

    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365
    df["EMPLOYMENT_YEARS"] = np.where(
        df["DAYS_EMPLOYED"] < 0,
        -df["DAYS_EMPLOYED"] / 365,
        np.nan
    )

    df["CREDIT_INCOME_RATIO"] = (
        df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    )

    df["ANNUITY_INCOME_RATIO"] = (
        df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    )

    df["CREDIT_GOODS_RATIO"] = (
        df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"]
    )

    return df

In [4]:
X_train = create_application_features(X_train)
X_val = create_application_features(X_val)
X_test = create_application_features(X_test)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (184506, 126)
Validation: (61502, 126)
Test: (61503, 126)


In [5]:
new_features = [
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_GOODS_RATIO"
]

X_train[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
AGE_YEARS,184506.0,43.916969,11.962367,20.517808,33.964384,43.134247,53.911644,69.120548
EMPLOYMENT_YEARS,151383.0,6.552964,6.437780,0.002740,2.109589,4.520548,8.706849,49.073973
CREDIT_INCOME_RATIO,184506.0,3.957286,2.680660,0.037500,2.018667,3.267273,5.163692,49.227200
ANNUITY_INCOME_RATIO,184498.0,0.180901,0.094360,0.003333,0.114726,0.163000,0.229080,1.570600
CREDIT_GOODS_RATIO,184352.0,1.122775,0.124081,0.150000,1.000000,1.118800,1.198000,6.000000


In [6]:
bureau = pd.read_csv("../data/bureau.csv")

print("Bureau shape:", bureau.shape)
print("Unique applicants:", bureau["SK_ID_CURR"].nunique())

Bureau shape: (1716428, 17)
Unique applicants: 305811


In [7]:
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_LOAN_COUNT=("SK_ID_BUREAU", "count"),
    BUREAU_ACTIVE_COUNT=("CREDIT_ACTIVE", lambda x: (x == "Active").sum()),
    BUREAU_CLOSED_COUNT=("CREDIT_ACTIVE", lambda x: (x == "Closed").sum()),
    BUREAU_CREDIT_SUM=("AMT_CREDIT_SUM", "sum"),
    BUREAU_CREDIT_SUM_MEAN=("AMT_CREDIT_SUM", "mean"),
    BUREAU_DEBT_SUM=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_OVERDUE_SUM=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    BUREAU_OVERDUE_MEAN=("AMT_CREDIT_SUM_OVERDUE", "mean"),
    BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean")
).reset_index()

print("Bureau aggregates:", bureau_agg.shape)
bureau_agg.head()

Bureau aggregates: (305811, 10)


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_CLOSED_COUNT,BUREAU_CREDIT_SUM,BUREAU_CREDIT_SUM_MEAN,BUREAU_DEBT_SUM,BUREAU_OVERDUE_SUM,BUREAU_OVERDUE_MEAN,BUREAU_DAYS_CREDIT_MEAN
0,100001,7,3,4,1453365.000,207623.571429,596686.5,0.0,0.0,-735.000000
1,100002,8,2,6,865055.565,108131.945625,245781.0,0.0,0.0,-874.000000
2,100003,4,1,3,1017400.500,254350.125000,0.0,0.0,0.0,-1400.750000
3,100004,2,0,2,189037.800,94518.900000,0.0,0.0,0.0,-867.000000
4,100005,3,2,1,657126.000,219042.000000,568408.5,0.0,0.0,-190.666667


In [8]:
X_train = X_train.merge(
    bureau_agg,
    on="SK_ID_CURR",
    how="left"
)

X_val = X_val.merge(
    bureau_agg,
    on="SK_ID_CURR",
    how="left"
)

X_test = X_test.merge(
    bureau_agg,
    on="SK_ID_CURR",
    how="left"
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (184506, 135)
Validation: (61502, 135)
Test: (61503, 135)


In [9]:
previous = pd.read_csv("../data/previous_application.csv")

print("Previous applications shape:", previous.shape)
print("Unique applicants:", previous["SK_ID_CURR"].nunique())

Previous applications shape: (1670214, 37)
Unique applicants: 338857


In [10]:
previous_agg = previous.groupby("SK_ID_CURR").agg(
    PREV_APP_COUNT=("SK_ID_PREV", "count"),
    PREV_APPROVED_COUNT=(
        "NAME_CONTRACT_STATUS",
        lambda x: (x == "Approved").sum()
    ),
    PREV_REFUSED_COUNT=(
        "NAME_CONTRACT_STATUS",
        lambda x: (x == "Refused").sum()
    ),
    PREV_CREDIT_MEAN=("AMT_CREDIT", "mean"),
    PREV_CREDIT_SUM=("AMT_CREDIT", "sum"),
    PREV_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
    PREV_APPLICATION_MEAN=("AMT_APPLICATION", "mean"),
    PREV_DOWN_PAYMENT_MEAN=("AMT_DOWN_PAYMENT", "mean"),
    PREV_GOODS_PRICE_MEAN=("AMT_GOODS_PRICE", "mean")
).reset_index()

print("Previous application aggregates:", previous_agg.shape)

previous_agg.head()

Previous application aggregates: (338857, 10)


,SK_ID_CURR,PREV_APP_COUNT,PREV_APPROVED_COUNT,PREV_REFUSED_COUNT,PREV_CREDIT_MEAN,PREV_CREDIT_SUM,PREV_ANNUITY_MEAN,PREV_APPLICATION_MEAN,PREV_DOWN_PAYMENT_MEAN,PREV_GOODS_PRICE_MEAN
0,100001,1,1,0,23787.00,23787.0,3951.000,24835.50,2520.0,24835.5
1,100002,1,1,0,179055.00,179055.0,9251.775,179055.00,0.0,179055.0
2,100003,3,3,0,484191.00,1452573.0,56553.990,435436.50,3442.5,435436.5
3,100004,1,1,0,20106.00,20106.0,5357.250,24282.00,4860.0,24282.0
4,100005,2,1,0,20076.75,40153.5,4813.200,22308.75,4464.0,44617.5


In [11]:
X_train = X_train.merge(
    previous_agg,
    on="SK_ID_CURR",
    how="left"
)

X_val = X_val.merge(
    previous_agg,
    on="SK_ID_CURR",
    how="left"
)

X_test = X_test.merge(
    previous_agg,
    on="SK_ID_CURR",
    how="left"
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (184506, 144)
Validation: (61502, 144)
Test: (61503, 144)


In [13]:
installments = pd.read_csv("../data/installments_payments.csv")

print("Installments shape:", installments.shape)
print("Unique applicants:", installments["SK_ID_CURR"].nunique())

Installments shape: (13605401, 8)
Unique applicants: 339587


In [14]:
installments_agg = installments.groupby("SK_ID_CURR").agg(
    INSTALLMENT_COUNT=("SK_ID_PREV", "count"),
    INSTALLMENT_PAYMENT_MEAN=("AMT_PAYMENT", "mean"),
    INSTALLMENT_PAYMENT_SUM=("AMT_PAYMENT", "sum"),
    INSTALLMENT_INSTALMENT_MEAN=("AMT_INSTALMENT", "mean"),
    INSTALLMENT_INSTALMENT_SUM=("AMT_INSTALMENT", "sum"),
    INSTALLMENT_DAYS_ENTRY_MEAN=("DAYS_ENTRY_PAYMENT", "mean"),
    INSTALLMENT_DAYS_INSTALMENT_MEAN=("DAYS_INSTALMENT", "mean")
).reset_index()

# Payment-to-installment ratio
installments_agg["INSTALLMENT_PAYMENT_RATIO"] = (
    installments_agg["INSTALLMENT_PAYMENT_SUM"] /
    installments_agg["INSTALLMENT_INSTALMENT_SUM"].replace(0, np.nan)
)

print("Installment aggregates:", installments_agg.shape)

installments_agg.head()

Installment aggregates: (339587, 9)


,SK_ID_CURR,INSTALLMENT_COUNT,INSTALLMENT_PAYMENT_MEAN,INSTALLMENT_PAYMENT_SUM,INSTALLMENT_INSTALMENT_MEAN,INSTALLMENT_INSTALMENT_SUM,INSTALLMENT_DAYS_ENTRY_MEAN,INSTALLMENT_DAYS_INSTALMENT_MEAN,INSTALLMENT_PAYMENT_RATIO
0,100001,7,5885.132143,41195.925,5885.132143,41195.925,-2195.000000,-2187.714286,1.0
1,100002,19,11559.247105,219625.695,11559.247105,219625.695,-315.421053,-295.000000,1.0
2,100003,25,64754.586000,1618864.650,64754.586000,1618864.650,-1385.320000,-1378.160000,1.0
3,100004,3,7096.155000,21288.465,7096.155000,21288.465,-761.666667,-754.000000,1.0
4,100005,9,6240.205000,56161.845,6240.205000,56161.845,-609.555556,-586.000000,1.0


In [15]:
X_train = X_train.merge(
    installments_agg,
    on="SK_ID_CURR",
    how="left"
)

X_val = X_val.merge(
    installments_agg,
    on="SK_ID_CURR",
    how="left"
)

X_test = X_test.merge(
    installments_agg,
    on="SK_ID_CURR",
    how="left"
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (184506, 152)
Validation: (61502, 152)
Test: (61503, 152)


In [16]:
print("Final feature counts:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTotal features:", X_train.shape[1])

Final feature counts:
Train: (184506, 152)
Validation: (61502, 152)
Test: (61503, 152)

Total features: 152


In [17]:
duplicate_columns = X_train.columns[
    X_train.columns.duplicated()
]

print("Duplicate columns:", len(duplicate_columns))
print(duplicate_columns.tolist())

Duplicate columns: 0
[]


In [18]:
joblib.dump(X_train, "../data/X_train_engineered.pkl")
joblib.dump(X_val, "../data/X_val_engineered.pkl")
joblib.dump(X_test, "../data/X_test_engineered.pkl")

print("Engineered datasets saved successfully!")

Engineered datasets saved successfully!


# Feature Engineering Summary

Additional applicant-level features were created from the application data and historical credit records.

The application-level features include age, employment duration, credit-to-income ratio, annuity-to-income ratio, and credit-to-goods-price ratio.

Historical information was aggregated from the Bureau, Previous Application, and Installment Payment datasets. Aggregation was performed at the applicant level using `SK_ID_CURR`, allowing historical records to be incorporated without increasing the number of applicant observations.

The final engineered dataset contains 152 features, consisting of the original application features and newly created application, credit-history, previous-application, and repayment-behavior features.

The engineered training, validation, and test datasets were saved for use during model training.